In [30]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_CET_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data)

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_CET_data.parquet


In [31]:
df.tail()

ID,From,Date,Hour,Demand_kWh,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72
str,"datetime[ms, UTC]",date,i32,f64,str,str,f64,f64,f64,f64,f64
"""Exp_5408""",2021-03-26 19:00:00 UTC,2021-03-26,20,1.224,null,null,null,7.5,4.7,3.9,4.6
"""Exp_5408""",2021-03-26 20:00:00 UTC,2021-03-26,21,1.668,null,null,null,7.3,4.9,4.0,4.6
"""Exp_5408""",2021-03-26 21:00:00 UTC,2021-03-26,22,1.034,null,null,null,7.5,5.2,4.1,4.7
"""Exp_5408""",2021-03-26 22:00:00 UTC,2021-03-26,23,1.349,null,null,null,7.4,5.4,4.1,4.8
"""Exp_5408""",2021-03-26 23:00:00 UTC,2021-03-27,0,2.073,null,null,null,7.1,5.6,4.2,4.8


In [32]:
# ==========================================
# 2. FEATURE ENGINEERING: TEMPORAL & WEATHER
# ==========================================
# Add calendar features (Day of Week, Weekend indicator, Sunny conditions proxy if available)
df = df.with_columns(
    Day_Of_Week = pl.col("Date").dt.weekday(), # 1 = Monday, 7 = Sunday
).with_columns(
    Is_Weekend = pl.when(pl.col("Day_Of_Week").is_in([6, 7])).then(1).otherwise(0).cast(pl.Int32),
    Is_Sunny = pl.when(pl.col("Temperature24") > pl.col("Temperature24").mean()).then(1).otherwise(0).cast(pl.Int32))

In [33]:
# ==========================================
# 3. TARGET ENGINEERING: EXTENDED FPI MODEL
# ==========================================
print("Engineering Flexibility Propensity Index (FPI) with weather/weekend interactions...")

# Step A: Estimate household baseline consumption on non-experiment weekdays by hour & temperature bin
df_baselines = (
    df
    .filter(pl.col("Experiment_price_NOK_kWh").is_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5 # Bin temperature by 5°C increments
    )
    .group_by(["ID", "Hour", "Temp_Bin"])
    .agg(
        Baseline_kWh = pl.col("Demand_kWh").mean()
    )
)

# Step B: Join baseline back to experiment days
df_experiment_days = (
    df
    .filter(pl.col("Experiment_price_NOK_kWh").is_not_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5
    )
    .join(df_baselines, on=["ID", "Hour", "Temp_Bin"], how="left")
    .with_columns(
        Baseline_kWh = pl.col("Baseline_kWh").fill_null(pl.col("Demand_kWh").mean())
    )
)

# Step C: Aggregate to Household x Experiment-Day Level including interaction terms
df_fpi_daily = (
    df_experiment_days
    .group_by(["ID", "Date", "Price_profile", "Price_level", "Is_Weekend"])
    .agg(
        Actual_Peak_Demand = pl.col("Demand_kWh").sum(),
        Base_Peak_Demand = pl.col("Baseline_kWh").sum(),
        Mean_Temp = pl.col("Temperature").mean(),
        Mean_Sunny = pl.col("Is_Sunny").mean()
    )
    .with_columns(
        # Flexibility Propensity Index formula capturing demand shift/reduction
        Flexibility_Propensity_Index = (
            (pl.col("Base_Peak_Demand") - pl.col("Actual_Peak_Demand")) / (pl.col("Base_Peak_Demand") + 1e-6) * 100
        ).clip(-50.0, 50.0),
        
        # Interaction terms for weekend temperature sensitivity
        Temp_Weekend_Interaction = pl.col("Mean_Temp") * pl.col("Is_Weekend")
    )
)

print(f"Computed FPI across {df_fpi_daily.height:,} household-experiment days.")

Engineering Flexibility Propensity Index (FPI) with weather/weekend interactions...
Computed FPI across 121,730 household-experiment days.


In [34]:
df_fpi_daily.head()

ID,Date,Price_profile,Price_level,Is_Weekend,Actual_Peak_Demand,Base_Peak_Demand,Mean_Temp,Mean_Sunny,Flexibility_Propensity_Index,Temp_Weekend_Interaction
str,date,str,str,i32,f64,f64,f64,f64,f64,f64
"""Exp_4202""",2021-03-23,"""P""","""2""",0,1.68,2.094318,3.5,0.0,19.782953,0.0
"""Exp_4385""",2021-02-18,"""B""","""15""",0,6.4,5.603333,-8.9,0.0,-14.217725,-0.0
"""Exp_1834""",2021-02-10,"""P""","""5""",0,1.7,0.691667,-10.3,0.0,-50.0,-0.0
"""Exp_3980""",2021-01-19,"""A""","""10""",0,3.73,3.823571,-1.8,0.0,2.447225,-0.0
"""Exp_277""",2021-01-11,"""P""","""2""",0,115.751,104.006548,-1.691304,0.869565,-11.292031,-0.0


In [35]:
# ==========================================
# 4. APPENDIX B: GENERATING TABLES & FIGURES
# ==========================================

# --- TABLE B1: Dataset Dimensionality & Observation Breakdown ---
table_b1 = (
    df
    .group_by(["ID"])
    .agg(
        Total_Rows = pl.len(),
        Unique_Households = pl.col("ID").n_unique()
    )
    .sort(["ID"])
)

print("\n=== Table B1: Dataset Dimensionality & Observation Breakdown ===")
print(table_b1)


=== Table B1: Dataset Dimensionality & Observation Breakdown ===
shape: (4_375, 3)
┌─────────┬────────────┬───────────────────┐
│ ID      ┆ Total_Rows ┆ Unique_Households │
│ ---     ┆ ---        ┆ ---               │
│ str     ┆ u32        ┆ u32               │
╞═════════╪════════════╪═══════════════════╡
│ Exp_1   ┆ 2784       ┆ 1                 │
│ Exp_10  ┆ 2784       ┆ 1                 │
│ Exp_100 ┆ 2784       ┆ 1                 │
│ Exp_101 ┆ 2784       ┆ 1                 │
│ Exp_102 ┆ 2784       ┆ 1                 │
│ …       ┆ …          ┆ …                 │
│ Exp_95  ┆ 2784       ┆ 1                 │
│ Exp_96  ┆ 2784       ┆ 1                 │
│ Exp_97  ┆ 2784       ┆ 1                 │
│ Exp_98  ┆ 2784       ┆ 1                 │
│ Exp_99  ┆ 2784       ┆ 1                 │
└─────────┴────────────┴───────────────────┘


In [38]:
# --- FIGURE B2: Diurnal Load Profiles (Experiment vs Non-Experiment) ---
df_diurnal = (
    df
    .with_columns(
        Is_Experiment = pl.col("Experiment_price_NOK_kWh").is_not_null()
    )
    .with_columns(
        Mean_Demand = pl.col("Demand_kWh").mean().over(["Hour", "Is_Experiment"])
    )
    .sort(["Hour", "Is_Experiment"])
    .to_pandas()
)

# 1. Define your custom names and colors
label_map = {False: "Non-Experiment Baseline (Control)", True: "Active Price Experiment Day"}
palette_dict = {"Non-Experiment Baseline (Control)": "#1b9e77", "Active Price Experiment Day": "#d95f02"}

# 2. Create a temporary column with the descriptive names
plot_data = df_diurnal.assign(Group=df_diurnal["Is_Experiment"].map(label_map))

In [39]:
fig, ax = plt.subplots(figsize=(10, 5))
#palette_dict = {False: "#1b9e77", True: "#d95f02"}

# 3. Plot using the new column for hue
sns.lineplot(
    data=plot_data, 
    x="Hour", 
    y="Mean_Demand", 
    hue="Group", 
    palette=palette_dict,
    marker="o", 
    linewidth=2.5,
    ax=ax
)

ax.set_title("Mean Diurnal Load Profile: Experiment Days vs. Non-Experiment Days")
ax.set_xlabel("Hour of Day (0–23)")
ax.set_ylabel("Mean Hourly Electricity Demand (kWh)")
ax.set_xticks(range(0, 24))

fig_path_b2 = os.path.join(parent, "Data", "Figure_B6_Diurnal_Load_Profiles.png")
plt.savefig(fig_path_b2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved Figure B6 to: {fig_path_b2}")

Saved Figure B6 to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B6_Diurnal_Load_Profiles.png


In [40]:
#type(df_diurnal)
#type(pl.from_pandas(df_diurnal))
df_diurnal_pol = pl.from_pandas(df_diurnal)

In [41]:
# ==========================================
# FACETED GRID: DIURNAL PROFILES BY PROFILE & LEVEL
# ==========================================
print("Aggregating diurnal data by Price Profile and Price Level...")

# 1. Prepare data with Price_profile and Price_level indicators
df_faceted_diurnal = (
    df_diurnal_pol
    .with_columns(
        Is_Experiment = pl.col("Experiment_price_NOK_kWh").is_not_null()
    )
    # Filter out null profiles/levels if any control rows lack them
    .filter(pl.col("Price_profile").is_not_null() & pl.col("Price_level").is_not_null())
    .group_by(["Hour", "Price_profile", "Price_level", "Is_Experiment"])
    .agg(
        Mean_Demand = pl.col("Demand_kWh").mean()
    )
    .sort(["Price_profile", "Price_level", "Hour", "Is_Experiment"])
    .to_pandas()
)

# 2. Set up FacetGrid
palette_dict = {False: "#1b9e77", True: "#d95f02"}

g = sns.FacetGrid(
    df_faceted_diurnal, 
    col="Price_profile", 
    row="Price_level", 
    hue="Is_Experiment", 
    palette=palette_dict,
    height=3, 
    aspect=1.2, 
    sharey=False  # Allow individual Y-scales since baseline demands vary by price tier
)

# 3. Map line plots onto the grid
g.map_dataframe(sns.lineplot, x="Hour", y="Mean_Demand", marker="o", linewidth=2)

# 4. Formatting and titles
g.set_axis_labels("Hour of Day (1–24)", "Mean Demand (kWh)")
g.add_legend(title="Is Experiment", loc="upper right")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Diurnal Load Profiles Across Price Profiles and Peak Price Levels")

# Save figure
fig_path_grid = os.path.join(parent, "Data", "Figure_B2_Faceted_Diurnal_Profiles.png")
g.savefig(fig_path_grid, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved faceted diurnal grid to: {fig_path_grid}")

Aggregating diurnal data by Price Profile and Price Level...
Saved faceted diurnal grid to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B2_Faceted_Diurnal_Profiles.png


In [50]:
# ==========================================
# FACETED GRID: TREATMENT VS. CONTROL OVERLAY BY PRICE PROFILE & LEVEL
# ==========================================

# 1. Get unique sorted lists of profiles and levels
profiles = sorted(df_diurnal_pol.filter(pl.col("Price_profile").is_not_null())["Price_profile"].unique().to_list())
levels = sorted(df_diurnal_pol.filter(pl.col("Price_level").is_not_null())["Price_level"].unique().to_list())

# 2. Set up the matplotlib subplot grid
fig, axes = plt.subplots(len(levels), len(profiles), figsize=(18, 20), sharex=True, sharey=False)

# Define label map and colors matching Figure B2 exactly
label_map = {False: "Non-Experiment Baseline (Control)", True: "Active Price Experiment Day"}
palette_dict = {
    "Non-Experiment Baseline (Control)": "#1b9e77", 
    "Active Price Experiment Day": "#d95f02"
}

# 3. Loop through each price level and profile combination
for r_idx, level in enumerate(levels):
    for c_idx, profile in enumerate(profiles):
        ax = axes[r_idx, c_idx] if len(levels) > 1 else axes[c_idx]
        
        # Filter treatment rows matching this specific profile & level
        df_exp_sub = (
            df_diurnal_pol
            .filter(
              #  (pl.col("Participation_status_Phase2") == "OK") &
              #  (pl.col("Experiment_price_NOK_kWh").is_not_null()) &
                (pl.col("Price_profile") == profile) &
                (pl.col("Price_level") == level)
            )
        )
        
        # Extract exact Date and Region keys as strings to guarantee join compatibility
        signal_keys = df_exp_sub.select([
            pl.col("Date").cast(pl.Utf8),
          # pl.col("Region").cast(pl.Utf8)
        ]).unique()
        
        # # Filter Control group matching those exact keys
        df_ctrl_sub = (
             df_diurnal_pol
        #     .filter(
        #         (pl.col("Participation_Phase") == "Phase_2") &
        #         (pl.col("Participation_status_Phase2") == "Control")
        #     )
             .with_columns([
                 pl.col("Date").cast(pl.Utf8),
        #         pl.col("Region").cast(pl.Utf8)
             ])
             .join(signal_keys, on=["Date"], how="inner")
         )
        
        # Aggregate Experiment hourly means
        df_exp_agg = (
            df_exp_sub
            .group_by("Hour")
            .agg(Mean_Demand = pl.col("Demand_kWh").mean())
            .with_columns(Is_Experiment = pl.lit(True))
        )
        
        # Aggregate Control hourly means
        df_ctrl_agg = (
            df_ctrl_sub
            .group_by("Hour")
            .agg(Mean_Demand = pl.col("Demand_kWh").mean())
            .with_columns(Is_Experiment = pl.lit(False))
        )
        
        # Combine into a single pandas dataframe for plotting
        sub_plot_data = (
            pl.concat([df_exp_agg, df_ctrl_agg])
            .sort(["Hour", "Is_Experiment"])
            .to_pandas()
        )
        
        # Map boolean to descriptive legend labels
        sub_plot_data["Group"] = sub_plot_data["Is_Experiment"].map(label_map)
        
        # Plot both lines if data exists
        if not sub_plot_data.empty:
            sns.lineplot(
                data=sub_plot_data,
                x="Hour",
                y="Mean_Demand",
                hue="Group",
                palette=palette_dict,
                marker="o",
                linewidth=2,
                ax=ax,
                legend=False  # Suppress per-subplot legends in favor of a clean global legend
            )
            
        ax.set_title(f"Level: {level} | Profile: {profile}", fontsize=14)
        ax.set_xlabel("Hour (1–24)" if r_idx == len(levels)-1 else "", fontsize=14)
        ax.set_ylabel("Demand (kWh)" if c_idx == 0 else "", fontsize=14)
        ax.set_xticks(range(0, 24, 5))
        ax.tick_params(axis="both", labelsize=12)

# 4. Add global legend handles and figure title matching Figure B2 styling
handles = [
    mlines.Line2D([], [], color="#1b9e77", marker='o', linewidth=2.5, label="Non-Experiment Baseline (Control)"),
    mlines.Line2D([], [], color="#d95f02", marker='o', linewidth=2.5, label="Active Price Experiment Day")
]
fig.legend(handles=handles, title="Experimental Condition", loc="upper right", bbox_to_anchor=(0.98, 0.99), fontsize=14, title_fontsize=14)
fig.suptitle("Diurnal Load Profiles Across All Price Profiles & Levels:\nTreatment vs. Control", fontsize=24, y=0.99)#, ha="right")

plt.tight_layout(rect=[0, 0, 1, 0.96])

# Save figure
fig_path_loop_grid = os.path.join(parent, "Data", "Figure_B5_Faceted_Control_Overlay_Grid.png")
plt.savefig(fig_path_loop_grid, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved faceted control overlay grid to: {fig_path_loop_grid}")

Saved faceted control overlay grid to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B5_Faceted_Control_Overlay_Grid.png
